# Day 5: Silver Standardization (OMOP CDM v5.4 & Entity Linkage)

This notebook demonstrates probabilistic patient linkage using Splink and mapping events to the OMOP CDM v5.4 `condition_occurrence` table.

In [ ]:
!pip install pyspark splink
import os

os.environ['SPARK_VERSION'] = '3.5'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
import splink.spark.linker as splink
from splink.spark.linker import SparkLinker
import splink.spark.comparison_library as cl

spark = SparkSession.builder.appName("Day05_OMOP_Linkage").getOrCreate()
print("Spark session created")

In [ ]:
# 1. Simulate Patient Linkage using Splink
data = [
    {"unique_id": "1", "first_name": "John", "last_name": "Smith", "dob": "1980-01-01", "city": "London"},
    {"unique_id": "2", "first_name": "Jon", "last_name": "Smith", "dob": "1980-01-01", "city": "London"},
    {"unique_id": "3", "first_name": "Jane", "last_name": "Doe", "dob": "1990-05-15", "city": "Paris"}
]
df = spark.createDataFrame(data)

settings = {
    "link_type": "dedupe_only",
    "comparisons": [
        cl.exact_match("first_name"),
        cl.exact_match("last_name"),
        cl.exact_match("dob"),
        cl.exact_match("city")
    ],
    "blocking_rules_to_generate_predictions": [
        "l.dob = r.dob"
    ],
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

linker = SparkLinker(df, settings)
linker.estimate_u_using_random_sampling(max_pairs=1e6)
linker.estimate_parameters_using_expectation_maximisation("l.dob = r.dob")

predictions = linker.predict(threshold_match_probability=0.8)
print("--- Splink Linkage Predictions ---")
predictions.as_pandas_dataframe().head()


In [ ]:
# 2. Simulate OMOP Mapping
# Assuming 'df' is the de-identified bronze dataframe
condition_occurrence_df = df \
    .select(
        col("unique_id").alias("condition_occurrence_id"),
        col("unique_id").alias("person_id"), # Typically replaced with linked person_id from Splink
        lit(0).alias("condition_concept_id"),
        col("dob").cast("date").alias("condition_start_date"),
        col("dob").cast("timestamp").alias("condition_start_datetime"),
        lit(None).cast("date").alias("condition_end_date"),
        lit(None).cast("timestamp").alias("condition_end_datetime"),
        lit(32020).alias("condition_type_concept_id"),
        col("city").alias("condition_source_value"),
        lit(0).alias("condition_source_concept_id"),
        lit(None).cast("string").alias("condition_status_source_value"),
        lit(0).alias("condition_status_concept_id")
    )

print("--- OMOP Condition Occurrence Table Structure ---")
condition_occurrence_df.show(truncate=False)
